# Gemma 4 31B QLoRA on SageMaker — notebook launcher

Notebook variant of `launch.py`. Pulls `HF_TOKEN` from Secrets Manager, then launches a single Training job that downloads the dataset, formats it for Gemma 4, and fine-tunes with QLoRA + Unsloth.

**Aligned on SageMaker Python SDK v3** (`sagemaker.train.ModelTrainer` + `sagemaker.core`).

## 1. Configuration

Replace `ROLE_ARN` with a SageMaker execution role ARN (find one with `aws iam list-roles --query "Roles[?contains(RoleName,'SageMaker-ExecutionRole')].Arn"`).

In [ ]:
REGION = "us-east-1"
ROLE_ARN = "arn:aws:iam::<account>:role/service-role/AmazonSageMaker-ExecutionRole-<id>"
SECRET_NAME = "huggingface/token"
SECRET_KEY = "HF_TOKEN"
DATASET = "mlabonne/guanaco-llama2-1k"
INSTANCE_TYPE = "ml.g6e.xlarge"   # bump to ml.g7e.2xlarge if MERGE = True
MAX_SEQ_LENGTH = 2048
MAX_RUN_SECONDS = 2 * 3600
VOLUME_SIZE_GB = 200
BUCKET_PREFIX = "gemma4-poc"

# If True, the training job also merges the adapter into a self-contained
# 4-bit checkpoint under <output>/merged_4bit/ — ready for direct vLLM/LMI
# serving with `option.quantization=bitsandbytes`. Requires ~96 GB GPU
# (ml.g7e.2xlarge); the script auto-bumps from ml.g6e.xlarge.
MERGE = False

if MERGE and INSTANCE_TYPE == "ml.g6e.xlarge":
    INSTANCE_TYPE = "ml.g7e.2xlarge"

# AWS HuggingFace training DLC (transformers 5.3.0 / pytorch 2.9.0 / cu130).
# SDK image_uris.retrieve doesn't know transformers 5.x yet; hardcode.
TRAINING_IMAGE = (
    f"763104351884.dkr.ecr.{REGION}.amazonaws.com/"
    f"huggingface-pytorch-training:2.9.0-transformers5.3.0-gpu-py312-cu130-ubuntu22.04"
)

## 2. Fetch HF_TOKEN from AWS Secrets Manager

Token never enters a notebook cell. The kernel's IAM identity needs `secretsmanager:GetSecretValue` on the secret's ARN.

In [ ]:
import json

import boto3

_secrets = boto3.client("secretsmanager", region_name=REGION)
_payload = json.loads(_secrets.get_secret_value(SecretId=SECRET_NAME)["SecretString"])
HF_TOKEN = _payload[SECRET_KEY]
print(f"HF_TOKEN loaded ({len(HF_TOKEN)} chars).")

## 3. SageMaker session

In [ ]:
from sagemaker.core.helper.session_helper import Session

assert "<account>" not in ROLE_ARN, "Replace ROLE_ARN in cell 2 before running."

boto_session = boto3.Session(region_name=REGION)
session = Session(boto_session=boto_session)
bucket = session.default_bucket()
print(f"Role: {ROLE_ARN}")
print(f"Bucket: s3://{bucket}")

## 4. Training job — Unsloth + QLoRA on Gemma 4 31B

Self-contained: `src/train.py` downloads the dataset, formats it for Gemma 4, fine-tunes, and saves the adapter. Container startup includes a 5–10 minute Unsloth git install — be patient.

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)

trainer = ModelTrainer(
    training_image=TRAINING_IMAGE,
    source_code=SourceCode(
        source_dir="src",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type=INSTANCE_TYPE,
        instance_count=1,
        volume_size_in_gb=VOLUME_SIZE_GB,
    ),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=MAX_RUN_SECONDS),
    hyperparameters={
        "model_id": "google/gemma-4-31B-it",
        "dataset": DATASET,
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora_rank": 64,
        "lora_alpha": 128,
        "lora_dropout": 0.05,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "num_train_epochs": 1,
        "learning_rate": 2e-4,
        "warmup_steps": 10,
        "merge": str(MERGE).lower(),
    },
    environment={"HF_TOKEN": HF_TOKEN, "TRANSFORMERS_CACHE": "/tmp/hf_cache"},
    output_data_config=OutputDataConfig(
        s3_output_path=f"s3://{bucket}/{BUCKET_PREFIX}/training/"
    ),
    role=ROLE_ARN,
    sagemaker_session=session,
    base_job_name="gemma4-unsloth-qlora",
)

try:
    trainer.train(wait=True, logs=True)
except Exception as e:
    msg = str(e).lower()
    if "resourcelimit" in msg or "capacity" in msg:
        print(f"\nCapacity/limit error on {INSTANCE_TYPE}: {e}\n"
              f"Try a different INSTANCE_TYPE (e.g. ml.g6e.2xlarge) or REGION.")
    raise

## 5. Adapter location

Adapter lands at `s3://<bucket>/<base-job-name>/output/model.tar.gz`. When `MERGE = True`, the tarball additionally contains a `merged_4bit/` directory ready for direct vLLM / DJL Serving deployment.

In [ ]:
_job_name = (
    getattr(trainer, "latest_training_job_name", None)
    or getattr(getattr(trainer, "_latest_training_job", None), "name", None)
)
_sm = boto3.client("sagemaker", region_name=REGION)
_train_desc = _sm.describe_training_job(TrainingJobName=_job_name) if _job_name else {}
print("Adapter:", _train_desc.get("ModelArtifacts", {}).get("S3ModelArtifacts"))
print("Billable seconds:", _train_desc.get("BillableTimeInSeconds"))